<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Generación masiva de informes PDF con LaTeX**
## *Plantilla + parámetros + compilación + auditoría*

Construiremos un sistema capaz de generar automáticamente un PDF por cliente, región o unidad de negocio.

```text
CSV
 ↓
validación
 ↓
Python
 ↓
Jinja2
 ↓
plantilla LaTeX
 ↓
.tex
 ↓
pdflatex / latexmk
 ↓
PDF
 ↓
auditoría
```

La separación entre **datos**, **plantilla** y **lógica** hace que el proyecto sea reutilizable.

## Objetivos

Al finalizar deberías poder:

1. Crear una plantilla LaTeX parametrizada.
2. Separar datos y presentación.
3. Usar Jinja2.
4. Escapar caracteres especiales de LaTeX.
5. Generar `.tex` en lote.
6. Implementar un modo dry-run.
7. Detectar `pdflatex` o `latexmk`.
8. Compilar desde Python.
9. Capturar errores de compilación.
10. Continuar aunque falle un informe.
11. Crear una auditoría.
12. Validar PDFs generados.

## 1. Preparación

In [ ]:
from pathlib import Path
import shutil
import subprocess
import time

import pandas as pd
import matplotlib.pyplot as plt

try:
    from jinja2 import Environment, FileSystemLoader, StrictUndefined
    JINJA_DISPONIBLE = True
except ImportError:
    JINJA_DISPONIBLE = False

RAIZ = Path("mini_proyecto_informes_pdf_latex")

print("Jinja2:", JINJA_DISPONIBLE)
print("pdflatex:", shutil.which("pdflatex"))
print("latexmk:", shutil.which("latexmk"))

## 2. Estructura esperada

In [ ]:
for ruta in sorted(RAIZ.rglob("*")):
    nivel = len(ruta.relative_to(RAIZ).parts) - 1
    simbolo = "📁" if ruta.is_dir() else "📄"
    print("    " * nivel + simbolo + " " + ruta.name)

## 3. Leer datos

In [ ]:
datos = pd.read_csv(
    RAIZ / "data" / "clientes.csv"
)

datos

## 4. Introducir inconsistencia

In [ ]:
datos_problematicos = datos.copy()

datos_problematicos.loc[
    2,
    "responsable"
] = None

datos_problematicos

## 4.1 Validar datos

In [ ]:
COLUMNAS_OBLIGATORIAS = [
    "cliente_id",
    "cliente",
    "periodo",
    "facturacion",
    "operaciones",
    "incidencias",
    "responsable"
]

def validar_datos(df):
    errores = []

    faltantes = [
        c
        for c in COLUMNAS_OBLIGATORIAS
        if c not in df.columns
    ]

    if faltantes:
        errores.append(
            f"Faltan columnas: {faltantes}"
        )

    if set(COLUMNAS_OBLIGATORIAS).issubset(df.columns):
        filas_na = df[
            COLUMNAS_OBLIGATORIAS
        ].isna().any(axis=1)

        if filas_na.any():
            errores.append(
                "Filas con NA: "
                + str(
                    df.index[
                        filas_na
                    ].tolist()
                )
            )

    return errores

validar_datos(
    datos_problematicos
)

## 4.2 Corregir antes de generar

In [ ]:
datos_limpios = (
    datos_problematicos
    .fillna({
        "responsable":
            "Responsable no asignado"
    })
)

print(
    validar_datos(
        datos_limpios
    )
)

## 5. Inspeccionar plantilla

In [ ]:
ruta_plantilla = (
    RAIZ
    / "templates"
    / "informe_cliente.tex.j2"
)

print(
    ruta_plantilla.read_text(
        encoding="utf-8"
    )[:1800]
)

### Interpretación

La plantilla contiene exclusivamente:

- estructura;
- texto común;
- placeholders.

Los datos específicos del cliente llegan desde Python.

## 6. Escapar caracteres especiales

In [ ]:
def latex_escape(texto):
    if texto is None:
        return ""

    texto = str(texto)

    reemplazos = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}"
    }

    for original, reemplazo in reemplazos.items():
        texto = texto.replace(
            original,
            reemplazo
        )

    return texto

print(
    latex_escape(
        "Cliente_A & Partner 50%"
    )
)

Sin esta función, datos reales con `%`, `_`, `&`, `$` o `#` pueden romper la compilación.

## 7. Generar interpretación automática por reglas

In [ ]:
def interpretar_cliente(
    facturacion,
    incidencias
):
    partes = []

    if facturacion >= 120000:
        partes.append(
            "La facturación se sitúa en un nivel alto para el periodo."
        )
    else:
        partes.append(
            "La facturación se sitúa por debajo del umbral de referencia."
        )

    if incidencias <= 5:
        partes.append(
            "El volumen de incidencias es reducido."
        )
    elif incidencias <= 15:
        partes.append(
            "El volumen de incidencias requiere seguimiento."
        )
    else:
        partes.append(
            "El número de incidencias es elevado y debería priorizarse su revisión."
        )

    return " ".join(
        partes
    )

datos_limpios["interpretacion"] = datos_limpios.apply(
    lambda fila:
        interpretar_cliente(
            fila["facturacion"],
            fila["incidencias"]
        ),
    axis=1
)

datos_limpios[
    [
        "cliente",
        "interpretacion"
    ]
]

Por ahora la interpretación es determinista. En el módulo de IA podremos sustituir esta función por un LLM.

## 8. Configurar Jinja2

In [ ]:
if JINJA_DISPONIBLE:

    entorno = Environment(
        loader=FileSystemLoader(
            RAIZ / "templates"
        ),
        undefined=StrictUndefined,
        autoescape=False
    )

    entorno.filters[
        "latex_escape"
    ] = latex_escape

    plantilla = entorno.get_template(
        "informe_cliente.tex.j2"
    )

    print(
        "Plantilla cargada."
    )
else:
    print(
        "Instalar Jinja2."
    )

## 9. Preparar parámetros

In [ ]:
def parametros_desde_fila(
    fila
):
    return {
        "cliente_id":
            fila["cliente_id"],

        "cliente":
            fila["cliente"],

        "periodo":
            fila["periodo"],

        "facturacion_fmt":
            f'{fila["facturacion"]:,.2f} EUR',

        "operaciones":
            int(
                fila["operaciones"]
            ),

        "incidencias":
            int(
                fila["incidencias"]
            ),

        "responsable":
            fila["responsable"],

        "interpretacion":
            fila["interpretacion"]
    }

## 10. Renderizar un informe

In [ ]:
if JINJA_DISPONIBLE:

    parametros = parametros_desde_fila(
        datos_limpios.iloc[0]
    )

    tex = plantilla.render(
        **parametros
    )

    print(
        tex[:1800]
    )

## 10.1 Guardar `.tex`

In [ ]:
if JINJA_DISPONIBLE:

    ruta_tex = (
        RAIZ
        / "outputs"
        / "tex"
        / "C001_2026-Q1_informe.tex"
    )

    ruta_tex.write_text(
        tex,
        encoding="utf-8"
    )

    print(
        ruta_tex
    )

### Dry-run

En proyectos reales conviene generar primero todos los `.tex` y revisarlos antes de compilar en lote.

## 11. Detectar motor LaTeX

In [ ]:
def detectar_motor():
    if shutil.which(
        "latexmk"
    ):
        return "latexmk"

    if shutil.which(
        "pdflatex"
    ):
        return "pdflatex"

    return None

motor = detectar_motor()

print(
    "Motor:",
    motor
)

## 12. Compilación reusable

In [ ]:
def compilar_tex(
    ruta_tex,
    carpeta_pdf,
    motor
):
    ruta_tex = Path(
        ruta_tex
    )

    carpeta_pdf = Path(
        carpeta_pdf
    )

    carpeta_pdf.mkdir(
        parents=True,
        exist_ok=True
    )

    if motor is None:
        return {
            "ok":
                False,
            "pdf":
                None,
            "mensaje":
                "No existe motor LaTeX instalado."
        }

    if motor == "latexmk":
        comando = [
            "latexmk",
            "-pdf",
            "-interaction=nonstopmode",
            "-halt-on-error",
            f"-outdir={carpeta_pdf}",
            str(ruta_tex)
        ]
    else:
        comando = [
            "pdflatex",
            "-interaction=nonstopmode",
            "-halt-on-error",
            "-output-directory",
            str(carpeta_pdf),
            str(ruta_tex)
        ]

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    ruta_pdf = (
        carpeta_pdf
        / f"{ruta_tex.stem}.pdf"
    )

    return {
        "ok":
            resultado.returncode == 0
            and ruta_pdf.exists(),

        "pdf":
            str(ruta_pdf)
            if ruta_pdf.exists()
            else None,

        "returncode":
            resultado.returncode,

        "salida":
            (
                resultado.stdout
                + "\n"
                + resultado.stderr
            )[-2500:]
    }

## 12.1 Compilar primer documento

In [ ]:
if JINJA_DISPONIBLE:

    resultado_compilacion = compilar_tex(
        ruta_tex,
        RAIZ / "outputs" / "pdf",
        motor
    )

    {
        k: v
        for k, v in resultado_compilacion.items()
        if k != "salida"
    }

Si el equipo no tiene LaTeX instalado, el notebook sigue siendo útil porque genera todos los `.tex` y deja constancia del problema.

## 13. Provocar error de compilación

In [ ]:
ruta_rota = (
    RAIZ
    / "outputs"
    / "tex"
    / "ejemplo_roto.tex"
)

ruta_rota.write_text(
    "\\documentclass{article}\n"
    "\\begin{document}\n"
    "\\textbf{Texto sin cerrar\n"
    "\\end{document}\n",
    encoding="utf-8"
)

print(
    ruta_rota.read_text(
        encoding="utf-8"
    )
)

## 13.1 Observar fallo

In [ ]:
if motor is not None:

    fallo = compilar_tex(
        ruta_rota,
        RAIZ / "outputs" / "pdf",
        motor
    )

    print(
        "OK:",
        fallo["ok"]
    )

    print(
        fallo["salida"][-1200:]
    )
else:
    print(
        "Sin motor LaTeX: ejemplo de compilación omitido."
    )

Un error de un informe no debería detener necesariamente los otros 99.

## 14. Función para generar `.tex` por fila

In [ ]:
def generar_tex_cliente(
    fila,
    plantilla,
    carpeta_tex
):
    carpeta_tex = Path(
        carpeta_tex
    )

    carpeta_tex.mkdir(
        parents=True,
        exist_ok=True
    )

    contenido = plantilla.render(
        **parametros_desde_fila(
            fila
        )
    )

    nombre = (
        f'{fila["cliente_id"]}_'
        f'{fila["periodo"]}_'
        f'informe.tex'
    )

    ruta = (
        carpeta_tex
        / nombre
    )

    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

    return ruta

## 15. Generación masiva `.tex`

In [ ]:
if JINJA_DISPONIBLE:

    rutas_tex = [
        generar_tex_cliente(
            fila,
            plantilla,
            RAIZ / "outputs" / "tex"
        )
        for _, fila
        in datos_limpios.iterrows()
    ]

    [
        ruta.name
        for ruta
        in rutas_tex
    ]

## 15.1 Validar dry-run

In [ ]:
if JINJA_DISPONIBLE:

    assert (
        len(rutas_tex)
        == len(datos_limpios)
    )

    assert all(
        ruta.exists()
        for ruta in rutas_tex
    )

    print(
        "Dry-run validado."
    )

## 16. Generación masiva PDF

In [ ]:
def generar_lote(
    dataframe,
    plantilla,
    carpeta_tex,
    carpeta_pdf,
    motor
):
    auditoria = []

    for _, fila in dataframe.iterrows():

        inicio = time.perf_counter()

        try:
            ruta_tex = generar_tex_cliente(
                fila,
                plantilla,
                carpeta_tex
            )

            compilacion = compilar_tex(
                ruta_tex,
                carpeta_pdf,
                motor
            )

            estado = (
                "ok"
                if compilacion["ok"]
                else "error"
            )

            mensaje = (
                ""
                if compilacion["ok"]
                else compilacion.get(
                    "mensaje",
                    compilacion.get(
                        "salida",
                        ""
                    )[-500:]
                )
            )

        except Exception as error:
            ruta_tex = None
            compilacion = {
                "pdf":
                    None
            }
            estado = "error"
            mensaje = str(
                error
            )

        auditoria.append({
            "cliente_id":
                fila["cliente_id"],

            "cliente":
                fila["cliente"],

            "tex":
                str(ruta_tex)
                if ruta_tex
                else None,

            "pdf":
                compilacion.get(
                    "pdf"
                ),

            "estado":
                estado,

            "mensaje":
                mensaje,

            "segundos":
                time.perf_counter()
                - inicio
        })

    return pd.DataFrame(
        auditoria
    )


if JINJA_DISPONIBLE:

    auditoria = generar_lote(
        datos_limpios,
        plantilla,
        RAIZ / "outputs" / "tex",
        RAIZ / "outputs" / "pdf",
        motor
    )

    auditoria

## 17. Visualizar estado del lote

In [ ]:
if JINJA_DISPONIBLE:

    estado = auditoria[
        "estado"
    ].value_counts()

    plt.figure(
        figsize=(7, 5)
    )

    plt.bar(
        estado.index,
        estado.values
    )

    plt.ylabel(
        "Informes"
    )

    plt.title(
        "Resultado de generación"
    )

    plt.tight_layout()
    plt.show()

## 18. Auditoría CSV

In [ ]:
if JINJA_DISPONIBLE:

    ruta_auditoria = (
        RAIZ
        / "logs"
        / "auditoria_generacion.csv"
    )

    auditoria.to_csv(
        ruta_auditoria,
        index=False
    )

    print(
        ruta_auditoria
    )

## 19. Validar PDFs existentes

In [ ]:
if JINJA_DISPONIBLE:

    filas_ok = auditoria[
        auditoria[
            "estado"
        ] == "ok"
    ]

    if len(
        filas_ok
    ) == 0:
        print(
            "No hay PDFs compilados en este entorno."
        )

    else:
        validacion = []

        for ruta_pdf in filas_ok[
            "pdf"
        ]:
            ruta_pdf = Path(
                ruta_pdf
            )

            validacion.append({
                "pdf":
                    ruta_pdf.name,
                "existe":
                    ruta_pdf.exists(),
                "bytes":
                    ruta_pdf.stat().st_size
                    if ruta_pdf.exists()
                    else 0
            })

        pd.DataFrame(
            validacion
        )

## 20. Arquitectura final

In [ ]:
pd.DataFrame({
    "componente": [
        "data/clientes.csv",
        "templates/informe_cliente.tex.j2",
        "Python/Jinja2",
        "outputs/tex/",
        "outputs/pdf/",
        "logs/auditoria_generacion.csv"
    ],
    "responsabilidad": [
        "Datos",
        "Diseño",
        "Renderizado",
        "Intermedio revisable",
        "Resultado final",
        "Trazabilidad"
    ]
})

## 21. Mini-proyecto final

Genera al menos **20 informes**.

Cada registro debe incluir:

- identificador;
- nombre;
- periodo;
- cuatro KPIs;
- responsable;
- interpretación.

El pipeline debe:

1. leer CSV;
2. validar;
3. detectar NAs;
4. corregir/rechazar;
5. cargar plantilla;
6. escapar LaTeX;
7. renderizar `.tex`;
8. ejecutar dry-run;
9. compilar;
10. capturar errores;
11. continuar;
12. auditar;
13. medir tiempos;
14. validar salidas.

## 22. Ejercicios finales

1. Crea una plantilla con cinco parámetros.
2. Renderízala con Jinja2.
3. Introduce `%`, `_` y `&`.
4. Implementa `latex_escape`.
5. Genera un `.tex` por fila.
6. Implementa dry-run.
7. Detecta motor LaTeX.
8. Compila un PDF.
9. Provoca un error LaTeX.
10. Captura el error.
11. Continúa el lote.
12. Crea auditoría.
13. Calcula tasa de éxito.
14. Valida tamaño de PDFs.
15. Crea una segunda plantilla.
16. Parametriza periodo.
17. Genera 50 informes.
18. Reto: construye un sistema reusable de reporting PDF.

## 23. Preguntas de reflexión

1. ¿Qué aporta LaTeX frente a Excel?
2. ¿Por qué separar datos y plantilla?
3. ¿Qué aporta Jinja2?
4. ¿Por qué escapar caracteres?
5. ¿Qué es dry-run?
6. ¿Por qué conservar `.tex`?
7. ¿Qué hacer si falla un informe?
8. ¿Qué debe contener la auditoría?
9. ¿Cómo incorporar gráficos?
10. ¿Cómo sustituir la interpretación por IA?

## 24. Cierre

Ahora tenemos dos vías complementarias:

```text
Excel → informe interactivo

LaTeX → PDF final reproducible
```

El siguiente laboratorio volverá a la **generación masiva de Excel parametrizado**.